# Inner transaction report

`teal_inner_txn_report.InnerTxnReport` walks every `itxn_begin → … →
itxn_submit` chain in a TEAL program and reports each contributing
`itxn_field F` with the consumed operand and (where statically known)
its literal value.

The boundary structure is computed by a CodeQL query
(`innerTxnFields.ql`) using a tightened version of
`InnerTransactionField.contributesToItxn` — only the *closest*
enclosing `(start, end)` pair survives, so a field in the first txn
of a grouped submit doesn't accidentally also pair with the outer
`(begin, submit)` skipping the `itxn_next`.

In [1]:
import os, sys
from pathlib import Path

os.environ.setdefault("CODEQL", "/home/argi/tools/codeql/codeql")

HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent.parent / "src" / "analysis"))

import tealql.tealtools.ssa as teal_ssa
from tealql.tealtools.inner_txn_report import InnerTxnReport

FIXTURES = HERE.parent.parent / "tests" / "tealtools" / "itxn_report"

## Three small fixtures

- `simple_pay/`: one `itxn_begin → 4 fields → itxn_submit`.
- `grouped/`: one group, two txns separated by `itxn_next` (a pay
  followed by an axfer).
- `branch_amount/`: a single `itxn_submit` reachable via two
  branches, each with its own `itxn_begin` — yields two alternative
  groups, one per branch.
- `phi_amount/`: a single `itxn_begin` / `itxn_submit`, but the `Amount` value comes from a phi over two branches that join *before* the `itxn_field` op — the report should render `Amount = {100 | 200}`.
- `phi_amount_indirect/`: same shape, but with an extra empty BB (`l_intermediate: b l_set_amount`) between the join and the `itxn_field` op — the consumed operand is now an `IndirectPhi` whose single arg is the `DirectPhi` at the intermediate BB. Confirms the resolver recurses through both phi kinds correctly.


In [2]:
for case in ("simple_pay", "grouped", "branch_amount", "phi_amount", "phi_amount_indirect"):
    print(f"### {case}")
    p = teal_ssa.SSAProgram(FIXTURES / case / "db")
    p.propagate_constants()
    print(InnerTxnReport(p).render())
    print()

### simple_pay


=== Inner-txn group  prog.teal  submit@L13  (1 txn) ===
  Txn 1  (itxn_begin@L4 → itxn_submit@L13)
    TypeEnum = 1  (set@L6)
    Receiver = txn Sender  (set@L8)
    Amount   = 100  (set@L10)
    Fee      = 0  (set@L12)


### grouped


=== Inner-txn group  prog.teal  submit@L22  (2 txn) ===
  Txn 1  (itxn_begin@L4 → itxn_next@L12)
    TypeEnum = 1  (set@L6)
    Receiver = txn Sender  (set@L8)
    Amount   = 50  (set@L10)
  Txn 2  (itxn_next@L12 → itxn_submit@L22)
    TypeEnum      = 4  (set@L14)
    AssetReceiver = txn Sender  (set@L16)
    XferAsset     = 7777  (set@L18)
    AssetAmount   = 1  (set@L20)


### branch_amount


=== Inner-txn group  prog.teal  submit@L29  (1 txn) ===
  Txn 1  (itxn_begin@L10 → itxn_submit@L29)
    TypeEnum = 1  (set@L12)
    Receiver = txn Sender  (set@L14)
    Amount   = 200  (set@L16)

=== Inner-txn group  prog.teal  submit@L29  (1 txn) ===
  Txn 1  (itxn_begin@L20 → itxn_submit@L29)
    TypeEnum = 1  (set@L22)
    Receiver = txn Sender  (set@L24)
    Amount   = 100  (set@L26)


### phi_amount


=== Inner-txn group  prog.teal  submit@L25  (1 txn) ===
  Txn 1  (itxn_begin@L6 → itxn_submit@L25)
    TypeEnum = 1  (set@L8)
    Receiver = txn Sender  (set@L10)
    Amount   = {100 | 200}  (set@L23)


### phi_amount_indirect


=== Inner-txn group  prog.teal  submit@L32  (1 txn) ===
  Txn 1  (itxn_begin@L12 → itxn_submit@L32)
    TypeEnum = 1  (set@L14)
    Amount   = {100 | 200}  (set@L30)




## Real contract: xgov

Running `propagate_constants` + `propagate_scratch_constants` first
gives the report enough resolution to surface the literal
`ApprovalProgram` / `ClearStateProgram` bytes for app-create itxns
and the `ConfigAssetUnitName` byte literal for the asset-create.

In [3]:
prog_xgov = teal_ssa.SSAProgram(HERE.parent.parent / "tests" / "dbs" / "xgov-db")
prog_xgov.propagate_constants()
prog_xgov.propagate_scratch_constants()
report = InnerTxnReport(prog_xgov)
print(f"{len(report)} submit-group(s)")
print()
print(report.render())

6 submit-group(s)

=== Inner-txn group  approval.teal  submit@L307  (1 txn) ===
  Txn 1  (itxn_begin@L298 → itxn_submit@L307)
    TypeEnum          = 6  (set@L300)
    ApprovalProgram   = 0x0820020001311b221240001d361a0080044c6bea7212400001003119221231182213104488001123433119221240000100311822124423438a00003100320912442343  (set@L302)
    ClearStateProgram = 0x08810043  (set@L304)
    Fee               = 0  (set@L306)

=== Inner-txn group  approval.teal  submit@L536  (1 txn) ===
  Txn 1  (itxn_begin@L525 → itxn_submit@L536)
    TypeEnum          = 6  (set@L527)
    Fee               = 0  (set@L529)
    OnCompletion      = 5  (set@L531)
    ApprovalProgram   = 0x068101  (set@L533)
    ClearStateProgram = 0x068101  (set@L535)

=== Inner-txn group  approval.teal  submit@L785  (1 txn) ===
  Txn 1  (itxn_begin@L775 → itxn_submit@L785)
    TypeEnum      = 6  (set@L777)
    ApplicationID = app_global_get(V#1@L778)  (set@L780)
    Fee           = 0  (set@L784)

=== Inner-txn group  approval.te

## Programmatic access

`report.groups` is a list of `InnerTxnGroup` dataclasses. Each
`InnerTxn.fields` is a list of `InnerTxnField`s; `field.operand` is
the typed SSA operand consumed at the `itxn_field` op (so you can
chain into `.defined_by` / `.uses` / `.const_value`), and
`field.possible_values()` flattens phis to a list of
literal-or-symbolic strings.

In [4]:
# Show the structure of one group from xgov.
g0 = report.groups[0]
print(f"file:        {g0.file}")
print(f"submit_line: {g0.submit_line}")
print(f"# of txns:   {len(g0.txns)}")
for txn in g0.txns:
    print(f"  txn {txn.begin_kind}@L{txn.begin_line} → {txn.end_kind}@L{txn.end_line}:")
    for f in txn.fields:
        print(f"    {f.name:20s} operand={f.operand!r:40s} → {f.possible_values()}")

file:        approval.teal
submit_line: 307
# of txns:   1
  txn itxn_begin@L298 → itxn_submit@L307:
    TypeEnum             operand=V#1@L299                                 → ['6']
    ApprovalProgram      operand=V#1@L301                                 → ['0x0820020001311b221240001d361a0080044c6bea7212400001003119221231182213104488001123433119221240000100311822124423438a00003100320912442343']
    ClearStateProgram    operand=V#1@L303                                 → ['0x08810043']
    Fee                  operand=V#1@L305                                 → ['0']


## Filter: only itxns that send an outbound call (`appl` or `pay`)

`InnerTxnGroup` is a plain dataclass — easy to query.

In [5]:
def has_field_value(group, name, value_str):
    return any(
        f.name == name and value_str in f.possible_values()
        for txn in group.txns for f in txn.fields
    )

appl_groups = [g for g in report.groups if has_field_value(g, "TypeEnum", "6")]
print(f"{len(appl_groups)} group(s) submit a TypeEnum=appl (6) inner txn:")
for g in appl_groups:
    print(f"  - submit@L{g.submit_line}")

5 group(s) submit a TypeEnum=appl (6) inner txn:
  - submit@L307
  - submit@L536
  - submit@L785
  - submit@L870
  - submit@L1193
